In [ ]:
#@title ⚙️ نصب و دانلود (pocket-tts-farsi-v2 + G2P فارسی)

HF_TOKEN = "" #@param {type:"string"}
USE_GPU = True #@param {type:"boolean"}

import os, sys, subprocess, shutil

if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()
    print('✅ توکن Hugging Face تنظیم شد.')
else:
    os.environ.pop('HF_TOKEN', None)
    print('ℹ️ بدون توکن Hugging Face ادامه می‌دهیم.')

print('=' * 40)

# ── ۱) PyTorch — با --index-url (نه --extra-index-url) تا نسخه‌ی CPU جایگزین نشود
if USE_GPU:
    print('⏳ نصب PyTorch (cu128) ...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'torch', 'torchaudio',
                        '--index-url', 'https://download.pytorch.org/whl/cu128'])
    if r.returncode != 0:
        print('⚠️ ایندکس cu128 ناموفق — نصب نسخه‌ی پیش‌فرض PyPI.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchaudio'], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchaudio'], check=True)

# ── ۲) pocket-tts از فورک مهدی الهیاری
# نسخه‌ی PyPI کلیدهای model.yaml این مدل را نمی‌شناسد و با
# ValidationError: capitalize_first_letter — Extra inputs are not permitted
# شکست می‌خورد. آن کلیدها عمداً خطا می‌دهند تا حرف اول کلمات بی‌صدا حذف نشود.
print('⏳ نصب pocket-tts (فورک فارسی) ...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pocket-tts @ git+https://github.com/mallahyari/pocket-tts@main'])
if r.returncode != 0:
    print('⚠️ نصب فورک ناموفق بود؛ تلاش دوباره با --no-deps ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                    'pocket-tts @ git+https://github.com/mallahyari/pocket-tts@main'], check=True)

# ── ۳) G2P و ابزارهای کمکی — بدون -U روی ipywidgets تا ویجت‌های Colab نشکنند
print('⏳ نصب transformers و ابزارها ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'sentencepiece', 'python-docx', 'pydub',
                'soundfile', 'scipy', 'ipywidgets'], check=True)

# ── ۴) ffmpeg
if shutil.which('ffmpeg') is None:
    print('⏳ نصب ffmpeg ...')
    subprocess.run('apt-get update -qq && apt-get install -y -qq ffmpeg', shell=True, check=True)

import torch
from pocket_tts import TTSModel

GPU_OK = torch.cuda.is_available() and USE_GPU
DEVICE = 'cuda' if GPU_OK else 'cpu'

if USE_GPU and not torch.cuda.is_available():
    print('⚠️ GPU در دسترس نیست — روی CPU اجرا می‌شود (این مدل روی CPU هم سریع است).')

if GPU_OK:
    props = torch.cuda.get_device_properties(0)
    print(f'🎮 GPU: {torch.cuda.get_device_name(0)} ({props.total_memory/1e9:.1f} GB)')
else:
    print('🖥️ اجرا روی CPU — pocket-tts برای CPU طراحی شده و ~۶ برابر سریع‌تر از زمان واقعی است.')

print('✅ نصب کامل شد. مدل‌ها در سلول بعدی لود می‌شوند.')


In [ ]:
#@title 🧠 لود مدل TTS و موتور G2P فارسی

import os, inspect, sys
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

# اگر سلول نصب اجرا نشده باشد، خطای واضح بدهد نه NameError
try:
    from pocket_tts import TTSModel
except ImportError as e:
    raise RuntimeError(
        '❌ pocket-tts نصب نیست. ابتدا سلول «⚙️ نصب و دانلود» را اجرا کنید.'
    ) from e

MODEL_REPO   = 'mehdi-hf/pocket-tts-farsi-v2'
MODEL_CONFIG = f'hf://{MODEL_REPO}/model.yaml'
G2P_REPO     = 'mehdi-hf/Homo-GE2PE-Persian-HF'

#@markdown `Temperature` تنها اهرم شباهت در سطح مدل است: پایین‌تر = نمونه‌برداری
#@markdown محافظه‌کارانه‌تر و صدای پایدارتر؛ بالاتر = تنوع بیشتر و انحراف بیشتر از مرجع.
#@markdown مقدار ۰.۳ مقدار خود مدل است. برای شباهت بیشتر ۰.۲ تا ۰.۲۵ را امتحان کنید.
Temperature   = 0.3  #@param {type:"slider", min:0.1, max:1.0, step:0.05}
EosThreshold  = -4.0 #@param {type:"number"}

# ── نرمال‌ساز فارسی: در خودِ ریپوی مدل منتشر شده (normalize_fa.py)
print('⏳ دریافت نرمال‌ساز فارسی ...')
from huggingface_hub import hf_hub_download
_norm_path = hf_hub_download(MODEL_REPO, 'normalize_fa.py')
_norm_dir = os.path.dirname(_norm_path)
if _norm_dir not in sys.path:
    sys.path.insert(0, _norm_dir)
from normalize_fa import normalize_for_model

# ── مدل TTS
print('⏳ لود مدل pocket-tts-farsi-v2 (۱۰۹.۵M پارامتر، ~۴۴۰ مگابایت) ...')

_sig = inspect.signature(TTSModel.load_model)
_kw = {}
if 'config' in _sig.parameters:
    _kw['config'] = MODEL_CONFIG
if 'temp' in _sig.parameters:
    _kw['temp'] = float(Temperature)
if 'eos_threshold' in _sig.parameters:
    _kw['eos_threshold'] = float(EosThreshold)

try:
    tts_model = TTSModel.load_model(**_kw)
except TypeError as e:
    print(f'⚠️ امضای load_model متفاوت است ({e}) — تلاش با آرگومان موضعی.')
    tts_model = TTSModel.load_model(MODEL_CONFIG)

if GPU_OK:
    tts_model = tts_model.to(DEVICE)
    print(f'🎮 مدل روی {DEVICE} منتقل شد.')

SAMPLE_RATE = getattr(tts_model, 'sample_rate', 24000)

# ── موتور G2P (نوشتار فارسی → فونم)
print('⏳ لود G2P (Homo-GE2PE، ~۳۳ مگابایت) ...')
g2p_tok = AutoTokenizer.from_pretrained(G2P_REPO)
g2p_model = T5ForConditionalGeneration.from_pretrained(G2P_REPO).eval().to(DEVICE)

# نگاشت نمادهای داخلی GE2PE به نوشتار مدل TTS
_TO_PHONEMES = str.maketrans({'/': 'a', 'a': 'A', '@': '?', '$': 'S', 'c': 'C'})

# ───────────────────────── الفبای فونم مدل
# واژگان ۴۰۰۰ تکه‌ای مدل فقط این نویسه‌ها را می‌شناسد. هر چیز دیگری
# به <unk> تبدیل می‌شود و مدل را از توزیع آموزش بیرون می‌برد.
#   ?  انسدادی چاکنایی (همزه) — نویسه‌ی میانیِ کلمه هم هست: so?Al
#   ;  ژ  (pa;uheSgar)
#   S  ش ،  C  چ ،  A  آ/الف کشیده
PHONEME_ALPHABET = set(' -;?ACSabdefghijklmnopqrstuvxyz')


def sanitise_phonemes(p):
    """هر نویسه‌ی بیرون از الفبای مدل را به فاصله تبدیل می‌کند."""
    bad = sorted({c for c in p if c not in PHONEME_ALPHABET})
    if bad:
        for c in bad:
            p = p.replace(c, ' ')
    return ' '.join(p.split())


def phonemise(text: str) -> str:
    """متن فارسی → فونم رومانیزه. خروجی خالی یعنی مدل چیزی نخواهد گفت."""
    if not text or not text.strip():
        return ''
    try:
        t = normalize_for_model(text)          # اعداد، ی/ك عربی، اعراب، نیم‌فاصله
    except ValueError:
        # نرمال‌ساز عمداً استثنا می‌اندازد وقتی متن کاملاً بیرون از الفبای فارسی است
        # (فقط علامت، فقط Latin، یا خالی). یک واحد خراب نباید کل سند را بکشد.
        return ''
    # «؟» فارسی در ورودی G2P را گیج می‌کند. توجه: علامت «?» در خروجی فونم
    # انسدادی چاکنایی است و باید بمانَد — اینجا فقط ورودی فارسی پاک می‌شود.
    t = t.replace('؟', '').replace('?', '')
    if not t.strip():
        return ''
    enc = g2p_tok([t], add_special_tokens=False, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = g2p_model.generate(**enc, num_beams=5, max_length=512, early_stopping=True)
    raw = g2p_tok.batch_decode(out, skip_special_tokens=True)[0].strip()
    # "1" پرچم داخلی اضافه‌ی (ezafe) است و خودش فونم نیست
    return sanitise_phonemes(raw.translate(_TO_PHONEMES).replace('1', ''))


# ───────────────────────── شمارش توکن واقعی مدل
# بودجه‌ی طول را باید با توکن‌های واقعی سنجید، نه با تعداد کلمه.
# میانگین اندازه‌گیری‌شده روی همین مدل: هر کلمه‌ی فونم ≈ ۱.۵۵ توکن.
try:
    _TOK = tts_model.flow_lm.conditioner.tokenizer
except AttributeError:
    _TOK = None


def count_tokens(phonemes):
    """تعداد توکن‌های sentencepiece مدل برای یک رشته‌ی فونم."""
    if not phonemes:
        return 0
    if _TOK is None:
        return len(phonemes.split()) * 2      # تخمین محافظه‌کارانه
    return int(_TOK(phonemes).shape[1])


def auto_frames_after_eos(text):
    """همان حدسی که خودِ کتابخانه می‌زند: ۳ فریم برای جمله‌ی کوتاه، ۱ برای بقیه، +۲.

    اگر این مقدار صفر پاس شود، حلقه‌ی `_generate` بلافاصله در همان فریم EOS
    می‌ایستد و دنباله‌ی واکه‌ی آخر بریده می‌شود. مقدار None یعنی «حدس بزن».
    """
    words = len(text.split())
    return (3 if words <= 4 else 1) + 2


def generate_exact(voice_state, phonemes, frames_after_eos=None):
    """تولید صدا بدون آنکه تقسیم‌کننده‌ی جمله‌ی کتابخانه به فونم دست بزند.

    `generate_audio` رشته را از `split_into_best_sentences` عبور می‌دهد و آن
    تابع روی نویسه‌های «.!...?» می‌شکند. در این مدل «?» انسدادی چاکنایی است و
    وسط کلمه هم می‌آید، پس کلمه‌هایی مثل so?Al به «so? Al» تبدیل می‌شدند.
    اندازه‌گیری روی ۶ جمله‌ی فارسی واقعی: هر ۶ جمله خراب می‌شد و ۲۳٪ کلمه‌های
    فونم همزه‌ی میانی داشتند. اینجا مستقیم به مسیر تولید می‌رویم.

    frames_after_eos=None یعنی حدس خودکار (رفتار پیش‌فرض کتابخانه).
    """
    text = ' '.join(phonemes.split())
    if not text:
        raise ValueError('رشته‌ی فونم خالی است')

    if frames_after_eos is None:
        frames_after_eos = auto_frames_after_eos(text)
    frames_after_eos = int(frames_after_eos)

    fn = getattr(tts_model, '_generate_audio_stream_short_text', None)
    if fn is None:
        # نسخه‌ی دیگری از کتابخانه — به مسیر عمومی برگرد
        return tts_model.generate_audio(
            voice_state, text, frames_after_eos=frames_after_eos)

    parts = []
    for piece in fn(model_state=voice_state, text_to_generate=text,
                    frames_after_eos=frames_after_eos, copy_state=True):
        parts.append(piece)
    if not parts:
        raise RuntimeError('هیچ صدایی تولید نشد')
    return torch.cat(parts, dim=0)


print()
print('✅ همه چیز آماده است.')
print(f'   TTS: {MODEL_REPO}  |  SR: {SAMPLE_RATE} Hz  |  device: {DEVICE}')
print(f'   G2P: {G2P_REPO}')
print(f'   شمارش توکن واقعی: {"فعال" if _TOK is not None else "تخمینی"}')
print()
print('🔍 نمونه‌ی تبدیل (توکن = توکن واقعی sentencepiece):')
for _s in ['سلام، حال شما چطور است؟', 'تأثیر مسئله بر مؤسسه', 'پژوهشگران دانشگاه تهران']:
    _p = phonemise(_s)
    print(f'   «{_s}»')
    print(f'   → {_p}   [{count_tokens(_p)} توکن]')


In [ ]:
#@title 📂 انتخاب فایل متنی (گوگل درایو یا حافظه داخلی)

import os

Use_Local_File = True #@param {type:"boolean"}

TEXT_EXTENSIONS = ('.txt', '.docx', '.md', '.rtf', '.srt', '.vtt', '.sub', '.ass')

SELECTED_DOC_PATH = None

if Use_Local_File:

    from google.colab import files

    print('📁 لطفاً فایل متنی خود را انتخاب کنید...')
    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError('❌ فایلی انتخاب نشد!')

    fname = list(uploaded.keys())[0]

    if not fname.lower().endswith(TEXT_EXTENSIONS):
        raise RuntimeError(
            '❌ این فرمت پشتیبانی نمی‌شود!\n'
            f'فرمت‌های مجاز: {", ".join(TEXT_EXTENSIONS)}'
        )

    SELECTED_DOC_PATH = os.path.abspath(fname)
    print(f'✅ فایل انتخاب شد: {SELECTED_DOC_PATH}')

else:

    from google.colab import drive
    import ipywidgets as widgets
    from IPython.display import display

    print('🔄 در حال اتصال به Google Drive...')
    drive.mount('/content/drive', force_remount=True)

    ROOT_DIR = '/content/drive/MyDrive'
    current_dir = [ROOT_DIR]

    path_label = widgets.HTML()
    listing = widgets.Select(rows=14, layout=widgets.Layout(width='100%'))
    btn_open = widgets.Button(description='📂 باز کردن / تأیید انتخاب', button_style='primary')
    btn_up = widgets.Button(description='⬆️ پوشه بالاتر')
    status = widgets.HTML()

    def refresh_listing():
        path_label.value = f'<b>📍 مسیر فعلی:</b> {current_dir[0]}'
        try:
            entries = sorted(os.listdir(current_dir[0]), key=lambda x: x.lower())
        except Exception as e:
            status.value = f'<span style="color:red">❌ خطا در خواندن پوشه: {e}</span>'
            listing.options = []
            return

        options = []
        for e in entries:
            full = os.path.join(current_dir[0], e)
            if os.path.isdir(full):
                options.append(f'📁 {e}')
            elif e.lower().endswith(TEXT_EXTENSIONS):
                options.append(f'📄 {e}')

        listing.options = options
        status.value = ('' if options else
                        '<span style="color:#888">📭 فایل متنی پشتیبانی‌شده‌ای در این پوشه نیست.</span>')

    def on_up_clicked(b):
        if os.path.normpath(current_dir[0]) == os.path.normpath(ROOT_DIR):
            status.value = '<span style="color:orange">⚠️ شما در پوشه اصلی Drive هستید.</span>'
            return
        parent = os.path.dirname(current_dir[0].rstrip('/'))
        if parent and len(parent) >= len(ROOT_DIR):
            current_dir[0] = parent
            refresh_listing()

    def on_open_clicked(b):
        global SELECTED_DOC_PATH

        if not listing.value:
            status.value = '<span style="color:orange">⚠️ ابتدا یک فایل انتخاب کنید.</span>'
            return

        name = listing.value[2:].strip()
        full = os.path.join(current_dir[0], name)

        if os.path.isdir(full):
            current_dir[0] = full
            refresh_listing()
            return

        if full.lower().endswith(TEXT_EXTENSIONS):
            SELECTED_DOC_PATH = full
            status.value = ('<div style="margin-top:8px;padding:10px;border-radius:6px;'
                            'background:#e8f5e9;color:#1b5e20;font-weight:bold;">'
                            f'✅ فایل تأیید شد.<br>📄 {name}</div>')
            print(f'✅ فایل انتخاب شد:\n{SELECTED_DOC_PATH}')

    btn_up.on_click(on_up_clicked)
    btn_open.on_click(on_open_clicked)

    refresh_listing()
    display(path_label, listing, widgets.HBox([btn_up, btn_open]), status)
    print('راهنما: پوشه‌ها را انتخاب کنید و «📂 باز کردن / تأیید انتخاب» را بزنید.')


In [ ]:
#@title 🎙️ صدای مرجع + موتور شباهت گوینده

from google.colab import files
import os, json, shutil, subprocess
import numpy as np
import torch

#@markdown **قاعده‌ی اول: پرامپت حداکثر ۵ ثانیه.** مدل روی پرامپت ≤۵s آموزش دیده؛
#@markdown بلندتر از آن باعث می‌شود مدل جمله‌ی خودِ پرامپت را ادامه دهد نه متن شما.
#@markdown ---
#@markdown #### 🎯 آماده‌سازی مرجع (بیشترین اثر را روی شباهت دارد)
#@markdown ۱) سکوت ابتدا/انتها بریده می‌شود — سکوت، هویت گوینده را رقیق می‌کند.
#@markdown آستانه **نسبی** است (سطح گفتار همین کلیپ منهای `Silence_Rel_dB`)،
#@markdown نه یک عدد ثابت؛ آستانه‌ی ثابت روی فایل‌های کم‌صداتر، گفتار واقعی را
#@markdown هم می‌بُرید.
#@markdown ۲) لرزش زیر ۷۰ هرتز و DC حذف می‌شود.
#@markdown ۳) بلندی صدا با **RMS** تنظیم می‌شود تا با فایل‌های پرامپت خود مدل بخواند.
#@markdown ۴) اگر فایل بلندتر از ۵ ثانیه باشد، پنجره‌ای با **بیشترین نسبت گفتار**
#@markdown انتخاب می‌شود (نه پنجره‌ای که تصادفاً سکوت بیشتری دارد).
Auto_Prepare_Reference = True #@param {type:"boolean"}
Voice_Prompt_Seconds = 5 #@param {type:"slider", min:3, max:8, step:1}
Silence_Rel_dB = 35 #@param {type:"slider", min:20, max:55, step:5}
Ref_Pad_Seconds = 0.15 #@param {type:"slider", min:0, max:0.5, step:0.05}
#@markdown ---
#@markdown #### 🔁 چند کلیپ از یک گوینده
#@markdown اگر ۲ تا ۳ کلیپ کوتاه از **همان گوینده** دارید، همه را با هم انتخاب کنید؛
#@markdown بردار مرجع از مرکز آن‌ها ساخته می‌شود و پایدارتر است، و کلیپی که
#@markdown به آن مرکز نزدیک‌ترین است برای voice state انتخاب می‌شود.
Use_Multi_Reference = False #@param {type:"boolean"}
Max_Reference_Clips = 3 #@param {type:"slider", min:1, max:4, step:1}
#@markdown ---
#@markdown #### 📏 سنجش شباهت (WavLM-SV)
#@markdown خروجی ۰ تا ۱؛ بالاتر = شبیه‌تر. همین مدل روی فارسی ~۰.۸۶ گزارش شده.
#@markdown سنجش، هزینه‌ی محاسباتی کمی دارد و برای انتخاب بهترین خروجی لازم است.
Measure_Similarity = True #@param {type:"boolean"}
#@markdown ---
#@markdown #### 🗣️ تنظیمات تولید
Max_Phoneme_Tokens = 24 #@param {type:"slider", min:8, max:40, step:1}
Frames_After_Eos = -1 #@param {type:"integer"}
Pause_Between_Segments_ms = 250 #@param {type:"slider", min:0, max:800, step:50}
#@markdown `Max_Phoneme_Tokens` اینجا **توکن واقعی sentencepiece** است، نه تعداد کلمه.
#@markdown سقف کتابخانه ۵۰ است؛ ۲۴ برای فارسی امن و روان است.
#@markdown `Frames_After_Eos = -1` یعنی «حدس خودکار» (۳ فریم برای جمله‌ی کوتاه،
#@markdown ۱ برای بقیه، به‌علاوه‌ی ۲). عدد ۰ این حدس را دور می‌زند و دنباله‌ی
#@markdown واکه‌ی آخر بریده می‌شود — فقط اگر عمداً می‌خواهید صفر بگذارید.
#@markdown `Temperature` در سلول «لود مدل» است — پایین‌تر = پایدارتر و شبیه‌تر.

# سقف سخت کتابخانه: MAX_TOKEN_PER_CHUNK = 50
HARD_TOKEN_LIMIT = 50
if Max_Phoneme_Tokens > 40:
    print('⚠️ بالای ۴۰ توکن به سقف ۵۰ کتابخانه نزدیک می‌شود؛ ممکن است کلمه بیفتد.')


# ───────────────────────── ابزار پایه
def probe_duration(path):
    ff = shutil.which('ffprobe')
    if not ff:
        return None
    r = subprocess.run([ff, '-v', 'error', '-show_entries', 'format=duration',
                        '-of', 'json', path], capture_output=True, text=True)
    try:
        return float(json.loads(r.stdout)['format']['duration'])
    except Exception:
        return None


def read_wav(path):
    """هر فرمت صوتی → numpy mono float32 با SAMPLE_RATE.

    mp3/m4a را scipy نمی‌خواند، پس برای آن‌ها از ffmpeg استفاده می‌شود.
    """
    import scipy.io.wavfile as wavfile
    from scipy.signal import resample_poly
    from math import gcd

    sr, a = None, None
    try:
        sr, a = wavfile.read(path)
        a = np.asarray(a)
        if a.ndim > 1:
            a = a.mean(axis=1)
        if a.dtype.kind == 'i':
            a = a.astype(np.float32) / max(1.0, float(np.iinfo(a.dtype).max))
        else:
            a = a.astype(np.float32)
    except (ValueError, OSError):
        ff = shutil.which('ffmpeg')
        if not ff:
            raise RuntimeError(
                f'خواندن «{os.path.basename(path)}» ممکن نشد و ffmpeg هم نصب نیست.'
            )
        r = subprocess.run(
            [ff, '-v', 'error', '-i', path, '-f', 'f32le', '-ac', '1',
             '-ar', str(SAMPLE_RATE), '-'],
            capture_output=True,
        )
        if r.returncode != 0 or not r.stdout:
            raise RuntimeError(
                f'ffmpeg نتوانست «{os.path.basename(path)}» را بخواند: '
                f'{r.stderr.decode("utf-8", "ignore")[:200]}'
            )
        a = np.frombuffer(r.stdout, dtype=np.float32).copy()
        sr = SAMPLE_RATE

    if sr != SAMPLE_RATE:
        g = gcd(int(sr), int(SAMPLE_RATE))
        a = resample_poly(a, SAMPLE_RATE // g, int(sr) // g).astype(np.float32)
    return a


def write_wav(path, a):
    import scipy.io.wavfile as wavfile
    a = np.clip(np.asarray(a, dtype=np.float32), -1.0, 1.0)
    wavfile.write(path, SAMPLE_RATE, (a * 32767).astype(np.int16))


def _frame_db(x, win_ms=25, hop_ms=10):
    win = max(1, int(SAMPLE_RATE * win_ms / 1000))
    hop = max(1, int(SAMPLE_RATE * hop_ms / 1000))
    c = np.concatenate([[0.0], np.cumsum(np.asarray(x, dtype=np.float64) ** 2)])
    starts = np.arange(0, max(1, len(x) - win), hop)
    starts = starts[starts + win < len(c)]
    if len(starts) == 0:
        return np.array([-120.0])
    e = (c[starts + win] - c[starts]) / win
    return 10 * np.log10(np.maximum(e, 1e-12))


def _speech_level_db(x):
    """سطح گفتار این کلیپ — صدک ۹۵ انرژی فریم‌ها (به سکوت حساس نیست)."""
    db = _frame_db(x)
    return float(np.percentile(db, 95)) if len(db) else -120.0


def _abs_threshold(x, rel_db, floor_db=-55.0):
    """آستانه‌ی مطلق از سطح گفتار همین کلیپ ساخته می‌شود، نه یک عدد ثابت.

    چرا نسبی: آستانه‌ی ثابت −۳۵ dB روی فایل‌های رسمی خود مدل، ۱.۳ ثانیه از
    انتهای prompt_hello را می‌برد که گفتار واقعی است. با آستانه‌ی نسبی
    (سطح گفتار منهای ۳۵ dB) همان فایل دست‌نخورده می‌ماند و در دو فایل دیگر
    فقط سکوت واقعی بریده می‌شود. اندازه‌گیری‌شده روی سه فایل رسمی:
        ثابت −۳۵ dB : 4.30→2.98 | 5.14→3.27 | 4.94→3.23
        نسبی ۳۵ dB  : 4.30→4.30 | 5.14→3.87 | 4.94→3.83
    """
    return max(_speech_level_db(x) - rel_db, floor_db)


def speech_fraction(x, thresh_db=None):
    """نسبت فریم‌های دارای گفتار — معیار «چقدر از این کلیپ حرف است»."""
    db = _frame_db(x)
    if len(db) == 0:
        return 0.0
    thr = _abs_threshold(x, 35.0) if thresh_db is None else thresh_db
    return float(np.mean(db > thr))


def trim_silence(x, thresh_db=None, pad_s=0.15, rel_db=35.0):
    """سکوت ابتدا و انتها را می‌برد. thresh_db=None یعنی آستانه‌ی نسبی."""
    if len(x) < 2000:
        return x
    db = _frame_db(x)
    hop = max(1, int(SAMPLE_RATE * 10 / 1000))
    thr = _abs_threshold(x, rel_db) if thresh_db is None else thresh_db
    idx = np.where(db > thr)[0]
    if len(idx) == 0:
        return x
    start = max(0, idx[0] * hop - int(pad_s * SAMPLE_RATE))
    end = min(len(x), idx[-1] * hop + int(SAMPLE_RATE * 0.025) + int(pad_s * SAMPLE_RATE))
    return x[start:end]


def rms_db(x):
    r = float(np.sqrt(np.mean(np.asarray(x, dtype=np.float64) ** 2))) if len(x) else 0.0
    return 20 * np.log10(max(r, 1e-12))


def match_loudness(x, target_db=-18.2, peak_ceiling_db=-1.5):
    """RMS را به سطح فایل‌های پرامپت خود مدل می‌برد، بعد سقف peak را رعایت می‌کند.

    چرا RMS و نه peak: فایل‌های پرامپت رسمی مدل RMS حدود −۱۸.۲ dBFS دارند.
    نرمال‌سازی روی peak، مقدار RMS را نامتوازن جابه‌جا می‌کند. اندازه‌گیری
    روی سه فایل رسمی خود مدل:
        peak-normalize : dRMS +۰.۵۵ / +۰.۹۸ / +۰.۹۲ dB
        RMS-match      : dRMS +۰.۵۵ / −۰.۶۷ / −۰.۰۴ dB
    """
    x = np.asarray(x, dtype=np.float32)
    cur = rms_db(x)
    if cur < -90:
        return x
    x = x * (10 ** ((target_db - cur) / 20))
    peak = float(np.max(np.abs(x))) if len(x) else 0.0
    ceiling = 10 ** (peak_ceiling_db / 20)
    if peak > ceiling:
        x = x * (ceiling / peak)
    return x.astype(np.float32)


def dc_and_rumble_filter(x, hp_hz=70.0):
    """حذف DC و لرزش زیر ۷۰ هرتز — این‌ها هویت گوینده نیستند، فقط پرامپت را خراب می‌کنند."""
    from scipy.signal import butter, sosfilt
    if len(x) < 1000:
        return x
    sos = butter(2, hp_hz / (SAMPLE_RATE / 2), btype='highpass', output='sos')
    return sosfilt(sos, x).astype(np.float32)


# ───────────────────────── موتور شباهت گوینده (WavLM-SV)
_SV_NAME = 'microsoft/wavlm-base-plus-sv'
_sv = {'fe': None, 'model': None, 'ref_emb': None, 'ready': False}


def load_similarity_model():
    """لود تنبل — فقط اگر لازم باشد. روی CPU سبک است (~۲۳۰MB)."""
    if _sv['ready'] or not Measure_Similarity:
        return _sv['ready']
    try:
        from transformers import AutoFeatureExtractor, AutoModelForAudioXVector
        _sv['fe'] = AutoFeatureExtractor.from_pretrained(_SV_NAME)
        _sv['model'] = AutoModelForAudioXVector.from_pretrained(_SV_NAME).eval()
        if GPU_OK:
            _sv['model'] = _sv['model'].to(DEVICE)
        _sv['ready'] = True
        print('✅ WavLM-SV آماده شد (سنجش شباهت گوینده).')
    except Exception as e:
        print(f'⚠️ سنجش شباهت غیرفعال شد ({type(e).__name__}: {e}).')
        _sv['ready'] = False
    return _sv['ready']


def speaker_embedding(audio, is_array=False):
    """بردار هویت گوینده (۵۱۲ بعد، نرمال‌شده). None یعنی در دسترس نیست."""
    if not load_similarity_model():
        return None
    from scipy.signal import resample_poly
    from math import gcd
    a = audio if is_array else read_wav(audio)
    a = np.asarray(a, dtype=np.float32)
    if len(a) < int(0.4 * SAMPLE_RATE):
        return None
    g = gcd(SAMPLE_RATE, 16000)
    a16 = resample_poly(a, 16000 // g, SAMPLE_RATE // g).astype(np.float32)
    a16 = a16[:16000 * 6]        # سقف ۶ ثانیه — کافی برای هویت گوینده، سبک برای حافظه
    inp = _sv['fe'](a16, sampling_rate=16000, return_tensors='pt').input_values
    if GPU_OK:
        inp = inp.to(DEVICE)
    with torch.inference_mode():
        emb = _sv['model'](inp).embeddings
    emb = torch.nn.functional.normalize(emb, dim=-1)
    return emb[0].float().cpu().numpy()


def cosine(u, v):
    if u is None or v is None:
        return None
    return float(np.dot(u, v))


def similarity_to_reference(audio, is_array=False):
    """شباهت گوینده‌ی این خروجی با مرجع (۰..۱). None = سنجش خاموش است."""
    if _sv['ref_emb'] is None:
        return None
    return cosine(speaker_embedding(audio, is_array=is_array), _sv['ref_emb'])


def pick_best_window(x, seconds, thresh_db=None, step_s=0.25, rel_db=35.0):
    """پنجره‌ی `seconds` ثانیه‌ای با **بیشترین نسبت گفتار**.

    تنها یک معیار، چون تنها یک معیار اندازه‌گیری‌شده نتیجه داد. دو معیار دیگر
    آزموده و رد شدند (اندازه‌گیری روی فایل ۱۴ ثانیه‌ای گوینده‌ی واحد، رتبه‌ی
    شباهت به هویت کل فایل از ۱۹ پنجره):
        پنجره‌ی اول                 رتبه ۲   (cos 0.9969)
        بیشترین نسبت گفتار          رتبه ۷   (cos 0.9602)
        مدوید بردار گوینده          رتبه ۷   (cos 0.9602)
        همبستگی نسبت‌گفتار↔شباهت  +0.080
    یعنی هیچ‌کدام «نماینده‌تر» نبودند. این تابع فقط تضمین می‌کند پنجره
    سکوت‌محور نباشد — که تنها چیزی است که واقعاً می‌شود ادعا کرد.
    """
    need = int(seconds * SAMPLE_RATE)
    if len(x) <= need + int(0.3 * SAMPLE_RATE):
        return x, 'کل فایل (کوتاه‌تر از پنجره)'

    step = max(1, int(step_s * SAMPLE_RATE))
    thr = _abs_threshold(x, rel_db) if thresh_db is None else thresh_db

    best, best_frac, n_win = None, -1.0, 0
    for st in range(0, len(x) - need + 1, step):
        c = x[st:st + need]
        f = speech_fraction(c, thr)
        n_win += 1
        if f > best_frac:
            best, best_frac = c, f
    if best is None:
        return x[:need], 'برش از ابتدا'

    return best, (f'بیشترین نسبت گفتار {best_frac:.2f} '
                  f'(از {n_win} پنجره، آستانه {thr:.0f} dBFS)')


def prepare_reference(x, target_seconds):
    """سکوت → فیلتر → انتخاب پنجره → تنظیم بلندی. ترتیب مهم است."""
    before = len(x) / SAMPLE_RATE
    if Auto_Prepare_Reference:
        x = trim_silence(x, None, Ref_Pad_Seconds, Silence_Rel_dB)
    trimmed = len(x) / SAMPLE_RATE

    x = dc_and_rumble_filter(x)

    if Auto_Prepare_Reference and len(x) > int(target_seconds * SAMPLE_RATE):
        x, why = pick_best_window(x, target_seconds, None, rel_db=Silence_Rel_dB)
    else:
        why = 'بدون برش'

    max_len = int(target_seconds * SAMPLE_RATE)
    if len(x) > max_len:
        x = x[:max_len]

    x = match_loudness(x)
    return x, before, trimmed, why


# ───────────────────────── آماده‌سازی کلیپ‌های مرجع
for f in os.listdir('.'):
    if f.startswith('ref_voice'):
        try:
            os.remove(f)
        except OSError:
            pass

n_want = int(Max_Reference_Clips) if Use_Multi_Reference else 1
hint = 'همان گوینده، هر کدام ۳ تا ۵ ثانیه، بدون موسیقی' if n_want > 1 else '۳ تا ۵ ثانیه، یک گوینده، بدون موسیقی'
print(f'📁 لطفاً {n_want} فایل صوتی صدای مرجع را آپلود کنید ({hint}):')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('❌ فایلی آپلود نشد!')

srcs = list(uploaded.keys())[:n_want]
REF_CLIPS, REF_ARRAYS = [], []

for i, src in enumerate(srcs):
    dur = probe_duration(src)
    print(f'\n[{i+1}/{len(srcs)}] {src}')
    if dur is not None:
        print(f'   ⏱️ طول: {dur:.2f} ثانیه')
        if dur < 3:
            print('   ⚠️ کوتاه‌تر از ۳ ثانیه — کلونینگ ضعیف می‌شود.')

    raw = f'ref_voice_raw_{i+1}.wav'
    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-i', src,
           '-ar', str(SAMPLE_RATE), '-ac', '1']
    if dur is not None and dur > Voice_Prompt_Seconds * 2:
        # سقف سخاوتمندانه؛ انتخاب پنجره در ادامه انجام می‌شود، نه برش کورکورانه
        cmd += ['-t', str(int(Voice_Prompt_Seconds * 2))]
    cmd.append(raw)
    subprocess.run(cmd, check=True)

    a = read_wav(raw)
    a, before, trimmed, why = prepare_reference(a, Voice_Prompt_Seconds)

    print(f'   ✂️ {before:.2f}s → پس از برش سکوت {trimmed:.2f}s → نهایی {len(a)/SAMPLE_RATE:.2f}s')
    print(f'   🎯 {why}')
    print(f'   🗣️ نسبت گفتار: {speech_fraction(a):.2f} | RMS: {rms_db(a):.1f} dBFS')

    clip_path = f'ref_voice_{i+1}.wav'
    write_wav(clip_path, a)
    REF_CLIPS.append(clip_path)
    REF_ARRAYS.append(a)

REF_AUDIO_PATH = REF_CLIPS[0]
REF_TIMBRE = np.concatenate(REF_ARRAYS)

# ───────────────────────── انتخاب کلیپ مرجع + voice state
# اگر چند کلیپ از یک گوینده دارید، برای سنجش شباهت از «مرکز» بردارهایشان
# استفاده می‌شود (پایدارتر از یک کلیپ تنها)، ولی خودِ voice state از **یک**
# کلیپ ساخته می‌شود — بافرهای KV مدل میانگین‌پذیر نیستند.
# کدام کلیپ؟ آن که بردارش به مرکز نزدیک‌ترین است، **فقط اگر** تفاوت معناداری
# باشد؛ وگرنه کلیپ اول. آستانه‌ی ۰.۰۲ عمداً محافظه‌کارانه است چون اندازه‌گیری
# نشان داد این معیار کوچک است و نباید زیاده از حد به آن تکیه کرد.
print('\n⏳ انتخاب کلیپ مرجع و استخراج voice state ...')

if Measure_Similarity:
    _clip_embs = [speaker_embedding(a, is_array=True) for a in REF_ARRAYS]
else:
    _clip_embs = []

if len(REF_CLIPS) > 1 and all(e is not None for e in _clip_embs):
    M = np.stack(_clip_embs)
    M = M / np.linalg.norm(M, axis=1, keepdims=True)
    centroid = M.mean(axis=0)
    centroid = centroid / np.linalg.norm(centroid)
    cen = M @ centroid
    medoid = int(np.argmax(cen))
    _sv['ref_emb'] = centroid.astype(np.float32)
    print(f'✅ مرکز هویت از {len(REF_CLIPS)} کلیپ ساخته شد.')
    for k in range(len(REF_CLIPS)):
        print(f'   • کلیپ {k + 1}: شباهت به مرکز {float(cen[k]):.3f}')
    if medoid != 0 and float(cen[medoid] - cen[0]) < 0.02:
        print('   ℹ️ تفاوت کلیپ‌ها ناچیز است — همان کلیپ اول استفاده می‌شود.')
        medoid = 0
    else:
        print(f'   → کلیپ {medoid + 1} (شباهت {float(cen[medoid]):.3f} به مرکز).')
    chosen = medoid
elif len(REF_CLIPS) > 1:
    print('ℹ️ سنجش شباهت در دسترس نیست — از کلیپ اول استفاده می‌شود.')
    chosen = 0
else:
    chosen = 0

try:
    voice_state = tts_model.get_state_for_audio_prompt(REF_CLIPS[chosen], truncate=True)
except TypeError:
    voice_state = tts_model.get_state_for_audio_prompt(REF_CLIPS[chosen])
print(f'✅ voice state از {REF_CLIPS[chosen]} آماده شد.')

# ───────────────────────── بردار مرجع نهایی
if Measure_Similarity:
    if _sv['ref_emb'] is None:
        print('\n⏳ محاسبه‌ی بردار هویت گوینده‌ی مرجع ...')
        _sv['ref_emb'] = speaker_embedding(REF_TIMBRE, is_array=True)
    if _sv['ref_emb'] is not None:
        print('✅ بردار مرجع آماده است — هر خروجی با آن سنجیده می‌شود.')
    else:
        print('⚠️ بردار مرجع ساخته نشد؛ تولید بدون سنجش شباهت ادامه می‌یابد.')

print(f'\n🎛️ سقف توکن هر تکه: {Max_Phoneme_Tokens} | سکوت بین تکه‌ها: {Pause_Between_Segments_ms}ms')


In [ ]:
#@title ▶️ شروع فرآیند: خواندن فایل با صدای کلون‌شده

import os, re, json, shutil, subprocess, traceback
import numpy as np
import torch
import scipy.io.wavfile as wavfile
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, Audio

#@markdown #### 🎯 انتخاب بهترین خروجی (مهم‌ترین اهرم شباهت)
#@markdown مدل برای هر تکه چند نمونه‌ی متفاوت تولید می‌کند و نمونه‌ای که
#@markdown **شباهت گوینده‌اش با مرجع بیشتر** است نگه داشته می‌شود.
#@markdown نمونه‌ها با هم تفاوت دارند چون سمپلینگ تصادفی است؛ بهترین از چند
#@markdown نمونه همیشه شبیه‌تر از تک‌نمونه است.
Best_of_N = 3 #@param {type:"slider", min:1, max:6, step:1}
#@markdown `Best_of_N = 1` یعنی رفتار قبلی (بدون انتخاب). ۳ تا ۴ تعادل خوبی است.
#@markdown ---
#@markdown #### 🎚️ تنظیمات
Similarity_Gate = 0.0 #@param {type:"slider", min:0.0, max:0.95, step:0.01}
#@markdown اگر شباهت بهترین نمونه از این عدد کمتر بود، تکه با هشدار ثبت می‌شود
#@markdown (۰ = بدون هشدار). برای گزارش کیفیت، نه توقف کار.
Edge_Fade_ms = 12 #@param {type:"slider", min:0, max:40, step:2}
#@markdown محو کوتاه در ابتدا/انتهای هر تکه، تا درزها کلیک نکنند.
Trim_Segment_Silence = True #@param {type:"boolean"}
#@markdown سکوت اضافی هر تکه پیش از چسباندن حذف می‌شود؛ مکث واقعی را
#@markdown `Pause_Between_Segments_ms` تعیین می‌کند، نه سکوت تصادفی مدل.
YOUTUBE_CHANNEL_URL = "https://www.youtube.com/@aigolden?sub_confirmation=1"

SUBTITLE_EXTS = ('.srt', '.vtt', '.sub', '.ass')
SILENCE_MS = int(Pause_Between_Segments_ms)
OUTPUT_FILE = 'cloned_voice_docx_reading.mp3'
SEG_DIR = 'voice_clone_segments'
FAIL_FILE = 'failed_paragraphs.txt'
REPORT_FILE = 'similarity_report.csv'

MIN_PHONEME_TOKENS = 6


def probe_duration(path):
    """مدت فایل صوتی به ثانیه (None اگر ffprobe نبود)."""
    ff = shutil.which('ffprobe')
    if not ff:
        return None
    r = subprocess.run([ff, '-v', 'error', '-show_entries', 'format=duration',
                        '-of', 'json', path], capture_output=True, text=True)
    try:
        return float(json.loads(r.stdout)['format']['duration'])
    except Exception:
        return None


# ───────────────────────── استخراج متن
def _clean_subtitle(raw):
    out = []
    for block in re.split(r'\n\s*\n', raw):
        lines = []
        for line in block.splitlines():
            s = line.strip()
            if not s:
                continue
            if s.isdigit():
                continue
            if '-->' in s:
                continue
            if re.match(r'^(WEBVTT|NOTE\b|STYLE\b|REGION\b)', s, re.I):
                continue
            if re.match(r'^\d{1,2}:\d{2}:\d{2}[.,]\d{3}\s*$', s):
                continue
            lines.append(s)
        if lines:
            out.append(' '.join(lines))
    return out


def _iter_docx_blocks(doc):
    from docx.document import Document as _Doc
    from docx.oxml.table import CT_Tbl
    from docx.oxml.text.paragraph import CT_P
    from docx.table import Table, _Cell
    from docx.text.paragraph import Paragraph

    def walk(parent):
        if isinstance(parent, _Doc):
            elm = parent.element.body
        elif isinstance(parent, _Cell):
            elm = parent._tc
        else:
            return
        for child in elm.iterchildren():
            if isinstance(child, CT_P):
                yield Paragraph(child, parent)
            elif isinstance(child, CT_Tbl):
                yield Table(child, parent)

    yield from walk(doc)


def extract_paragraphs(path):
    ext = path.rsplit('.', 1)[-1].lower()

    if ext == 'docx':
        import docx
        from docx.table import Table

        doc = docx.Document(path)
        out = []

        for block in _iter_docx_blocks(doc):
            if isinstance(block, Table):
                for row in block.rows:
                    cells, seen = [], set()
                    for cell in row.cells:
                        if id(cell._tc) in seen:
                            continue
                        seen.add(id(cell._tc))
                        t = ' '.join(p.text.strip() for p in cell.paragraphs if p.text.strip())
                        if t:
                            cells.append(t)
                    if cells:
                        out.append(' ، '.join(cells))
            else:
                t = block.text.strip()
                if t:
                    out.append(t)
        return out

    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()

    if f'.{ext}' in SUBTITLE_EXTS:
        return _clean_subtitle(raw)

    out = []
    for block in re.split(r'\n\s*\n', raw):
        lines = [l.strip() for l in block.splitlines() if l.strip()]
        if lines:
            out.append(' '.join(lines))
    return out


def merge_short(paras, min_len=25):
    merged = []
    for p in paras:
        if merged and len(p) < min_len:
            merged[-1] = merged[-1].rstrip() + ' ' + p
        else:
            merged.append(p)
    return merged


# ───────────────────────── تکه‌بندی
# G2P نشانه‌گذاری را دور می‌ریزد، پس باید *قبل* از فونم‌سازی تقسیم کرد؛
# وگرنه هر تکه‌بندی فقط روی تعداد توکن می‌برد و وسط عبارت می‌شکند.
def split_units(text):
    """جمله و عبارت را جدا می‌کند تا مکث‌ها سرِ جای خودشان بیفتند."""
    units = []
    for sent in re.split(r'(?<=[.!؟])\s+', text):
        for clause in re.split(r'(?<=[،؛:])\s+', sent):
            c = clause.strip()
            if c:
                units.append(c)
    return units or [text.strip()]


def _hard_split(text, ph, max_tokens):
    """یک واحدِ بلندتر از سقف را به تکه‌های **هم‌اندازه** می‌شکند.

    تقسیم فقط روی مرز کلمه‌های فونم انجام می‌شود و متن فارسی به همان نسبت
    بریده می‌شود؛ هیچ نویسه‌ای به فونم اضافه یا از آن حذف نمی‌شود.

    چرا هم‌اندازه و نه حریصانه: با برش حریصانه یک واحد ۲۵ توکنی با سقف ۲۴
    به «۲۴ + ۱» تبدیل می‌شد و تکه‌ی ۱ توکنی باقی می‌ماند — و تکه‌ی خیلی کوتاه
    باعث می‌شود مدل به‌جای خواندن متن، پرامپت را ادامه دهد.
    """
    tw, pw = text.split(), ph.split()
    if len(pw) <= 1:
        return [(text, ph)]

    n_tok = count_tokens(ph)
    k = max(2, -(-n_tok // max_tokens))       # ceil
    while k > 2 and count_tokens(' '.join(pw[:max(1, len(pw) // k)])) * k < n_tok - max_tokens:
        k -= 1

    out, i = [], 0
    for piece in range(k):
        remaining_pieces = k - piece
        if remaining_pieces == 1:
            j = len(pw)
        else:
            # هدف: سهم مساوی؛ ولی سقف مطلق همان بودجه است
            target = min(n_tok / k, max_tokens)
            j, n = i, 0
            while j < len(pw) - (remaining_pieces - 1):
                t = count_tokens(pw[j])
                if n and (n >= target or n + t > max_tokens):
                    break
                n += t
                j += 1
            j = min(max(j, i + 1), len(pw) - (remaining_pieces - 1))
        a = round(i * len(tw) / len(pw))
        b = max(a + 1, round(j * len(tw) / len(pw)))
        out.append((' '.join(tw[a:b]), ' '.join(pw[i:j])))
        i = j
    return out


def build_chunks(paragraph, budget=None):
    """پاراگراف فارسی → لیست (متن، فونم) با طول کنترل‌شده به **توکن واقعی**.

    چرا توکن و نه کلمه: شمارش کلمه‌ی فونم با توکن sentencepiece یکی نیست.
    اندازه‌گیری روی همین مدل: هر کلمه‌ی فونم به‌طور میانگین ~۱.۵۵ توکن است،
    پس سقف «۱۸ کلمه» در واقع تا ~۲۸ توکن می‌شد و مرز ۵۰تایی کتابخانه را
    برای جمله‌های بلند رد می‌کرد.
    """
    budget = int(budget or Max_Phoneme_Tokens)
    budget = max(4, min(budget, HARD_TOKEN_LIMIT))

    chunks, cur_text, cur_ph, cur_n = [], [], [], 0

    for unit in split_units(paragraph):
        try:
            ph = phonemise(unit)
        except Exception as e:
            print(f'   ⚠️ فونم‌سازی ناموفق: «{unit[:40]}» — {type(e).__name__}: {e}')
            continue
        ph = sanitise_phonemes(ph or '')
        if not ph:
            continue

        if count_tokens(ph) <= budget:
            pieces = [(unit, ph)]
        else:
            pieces = _hard_split(unit, ph, budget)

        for txt, p in pieces:
            n = count_tokens(p)
            if cur_n and cur_n + n > budget:
                chunks.append((' '.join(cur_text), ' '.join(cur_ph)))
                cur_text, cur_ph, cur_n = [], [], 0
            cur_text.append(txt)
            cur_ph.append(p)
            cur_n += n

    if cur_text:
        chunks.append((' '.join(cur_text), ' '.join(cur_ph)))

    # هر تکه‌ی خیلی کوتاه را به همسایه بچسبان. مدل روی جمله‌های کوتاه،
    # به‌جای خواندن متن، جمله‌ی پرامپت را ادامه می‌دهد.
    chunks = _merge_short_chunks(chunks, budget)
    return chunks


def _merge_short_chunks(chunks, budget):
    """تکه‌های زیر MIN_PHONEME_TOKENS توکن را با همسایه ادغام می‌کند.

    اولویت با همسایه‌ی قبلی است، ولی اگر ادغام از سقف رد شود و همسایه‌ی
    بعدی جا داشته باشد، با آن ادغام می‌شود. اگر هیچ‌کدام جا نداشتند،
    تکه همان‌طور می‌ماند (بهتر از رد کردن کلمه).
    """
    if len(chunks) < 2:
        return chunks

    out = [list(chunks[0])]
    for txt, ph in chunks[1:]:
        n_prev = count_tokens(out[-1][1])
        n_cur = count_tokens(ph)
        if (count_tokens(out[-1][1]) < MIN_PHONEME_TOKENS
                and n_prev + n_cur <= HARD_TOKEN_LIMIT):
            out[-1][0] = out[-1][0] + ' ' + txt
            out[-1][1] = out[-1][1] + ' ' + ph
        else:
            out.append([txt, ph])

    # نوبت آخری که ممکن است کوتاه مانده باشد
    if len(out) >= 2 and count_tokens(out[-1][1]) < MIN_PHONEME_TOKENS:
        n_prev = count_tokens(out[-2][1])
        n_last = count_tokens(out[-1][1])
        if n_prev + n_last <= HARD_TOKEN_LIMIT:
            t, p = out.pop()
            out[-1][0] = out[-1][0] + ' ' + t
            out[-1][1] = out[-1][1] + ' ' + p

    return [(t, p) for t, p in out]


# ───────────────────────── تولید صدا
def _to_numpy(audio):
    a = audio.detach().cpu().numpy() if isinstance(audio, torch.Tensor) else np.asarray(audio)
    return np.asarray(a, dtype=np.float32).ravel()


def _frames_arg():
    """-1 یعنی خودکار (None)، هر عدد دیگری صریح پاس می‌شود."""
    v = int(Frames_After_Eos)
    return None if v < 0 else v


def synth_once(phonemes):
    """یک نمونه تولید می‌کند و آن را به‌عنوان آرایه برمی‌گرداند (runaway → استثنا)."""
    audio = generate_exact(voice_state, phonemes, frames_after_eos=_frames_arg())
    a = _to_numpy(audio)

    dur = len(a) / SAMPLE_RATE
    expected = count_tokens(phonemes) / 3.0 + 2.0

    if dur > expected * 2.2:
        raise RuntimeError(
            f'runaway: {dur:.1f}s تولید شد در حالی که انتظار ~{expected:.1f}s بود'
        )
    if dur < 0.25:
        raise RuntimeError(f'خروجی تقریباً خالی است ({dur:.2f}s)')

    return np.clip(a, -1.0, 1.0)


def trim_tail_only(a, rel_db=30.0, pad_s=0.05, min_keep_s=0.15):
    """فقط سکوت انتهای تکه را می‌برد — ابتدا دست نمی‌خورد.

    چرا فقط انتها: حرف انسدادی آغازین ضعیف‌ترین نقطه‌ی مدل است؛ هر برشی از
    ابتدا می‌تواند انفجار اولیه را نصف کند. انتها اما فقط سکوت EOS است و
    نبودنش باعث می‌شود مکث‌های تراک نامنظم شود.
    """
    if len(a) < int(0.3 * SAMPLE_RATE):
        return a
    db = _frame_db(a)
    if len(db) == 0:
        return a
    hop = max(1, int(SAMPLE_RATE * 10 / 1000))
    thr = max(float(np.percentile(db, 95)) - rel_db, -60.0)
    idx = np.where(db > thr)[0]
    if len(idx) == 0:
        return a
    end = min(len(a), idx[-1] * hop + int(pad_s * SAMPLE_RATE))
    end = max(end, int(min_keep_s * SAMPLE_RATE))
    return a[:end]


def synth(phonemes, out_path=None, n_candidates=None):
    """تولید با انتخاب بهترین نمونه بر اساس شباهت گوینده.

    خروجی: (آرایه صوتی، شباهت بهترین نمونه یا None، تعداد نمونه‌های سالم)
    """
    n = max(1, int(n_candidates or Best_of_N))

    if n == 1 or _sv.get('ref_emb') is None:
        a = synth_once(phonemes)
        sim = similarity_to_reference(a, is_array=True)
        if out_path:
            write_wav(out_path, a)
        return a, sim, 1

    best_a, best_sim = None, -2.0
    got = 0
    for k in range(n):
        try:
            cand = synth_once(phonemes)
        except Exception:
            if got == 0 and k == n - 1:
                raise
            continue
        got += 1
        s = similarity_to_reference(cand, is_array=True)
        if s is None:
            s = 0.0
        if s > best_sim:
            best_a, best_sim = cand, s

    if best_a is None:
        raise RuntimeError('هیچ نمونه‌ی سالمی تولید نشد')

    if out_path:
        write_wav(out_path, best_a)
    return best_a, (best_sim if got else None), got


def _fade(a, ms):
    """محو کوتاه در دو سر تکه، تا درزها کلیک نکنند."""
    if ms <= 0 or len(a) < 8:
        return a
    n = min(int(SAMPLE_RATE * ms / 1000), len(a) // 2)
    if n <= 0:
        return a
    ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
    a = a.copy()
    a[:n] *= ramp
    a[-n:] *= ramp[::-1]
    return a


def concat_segments(seg_paths, out_path, gap_ms=SILENCE_MS, work_dir=None):
    if not seg_paths:
        raise RuntimeError('هیچ سگمنتی تولید نشد!')

    work_dir = work_dir or SEG_DIR
    os.makedirs(work_dir, exist_ok=True)

    silence = os.path.join(work_dir, '_silence.wav')
    ff = shutil.which('ffmpeg')

    if ff:
        subprocess.run(
            [ff, '-y', '-loglevel', 'error', '-f', 'lavfi',
             '-i', f'anullsrc=r={SAMPLE_RATE}:cl=mono', '-t', str(gap_ms / 1000), silence],
            check=True,
        )
        lst = os.path.join(work_dir, '_list.txt')
        with open(lst, 'w', encoding='utf-8') as f:
            for i, p in enumerate(seg_paths):
                f.write(f"file '{os.path.abspath(p)}'\n")
                if i != len(seg_paths) - 1 and gap_ms > 0:
                    f.write(f"file '{os.path.abspath(silence)}'\n")

        r = subprocess.run(
            [ff, '-y', '-loglevel', 'error', '-f', 'concat', '-safe', '0',
             '-i', lst, '-ar', str(SAMPLE_RATE), '-ac', '1',
             '-c:a', 'libmp3lame', '-b:a', '192k', out_path],
            capture_output=True, text=True,
        )
        if r.returncode == 0 and os.path.exists(out_path):
            return out_path
        print(f'⚠️ ffmpeg ناموفق بود ({r.stderr.strip()[:200]}) — از pydub استفاده می‌کنم.')

    from pydub import AudioSegment
    final = AudioSegment.empty()
    for i, p in enumerate(seg_paths):
        final += AudioSegment.from_file(p)
        if i != len(seg_paths) - 1 and gap_ms > 0:
            final += AudioSegment.silent(duration=gap_ms)
    final.export(out_path, format='mp3', bitrate='192k')
    return out_path


# ───────────────────────── اجرا
if 'SELECTED_DOC_PATH' not in globals() or not SELECTED_DOC_PATH:
    raise RuntimeError('❌ ابتدا در سلول «انتخاب فایل متنی» یک فایل انتخاب کنید!')
if 'voice_state' not in globals():
    raise RuntimeError('❌ ابتدا سلول «آماده‌سازی صدای مرجع» را اجرا کنید!')

print(f'📄 خواندن متن از: {SELECTED_DOC_PATH}')
PARAGRAPHS = merge_short(extract_paragraphs(SELECTED_DOC_PATH))

if not PARAGRAPHS:
    raise RuntimeError('❌ متنی در فایل یافت نشد!')

print(f'✅ {len(PARAGRAPHS)} پاراگراف ({sum(len(p) for p in PARAGRAPHS)} کاراکتر).')
print('⏳ فونم‌سازی و تکه‌بندی ...')

JOB_QUEUE = []
for pi, para in enumerate(PARAGRAPHS):
    for ci, (txt, ph) in enumerate(build_chunks(para)):
        JOB_QUEUE.append({'para': pi, 'chunk': ci, 'text': txt, 'ph': ph})

if not JOB_QUEUE:
    raise RuntimeError('❌ هیچ فونم معتبری تولید نشد — متن احتمالاً فارسی نیست.')

_tok_counts = [count_tokens(j['ph']) for j in JOB_QUEUE]
_soft = min(Max_Phoneme_Tokens, HARD_TOKEN_LIMIT)
print(f'✅ {len(JOB_QUEUE)} تکه برای سنتز آماده شد.')
print(f'   توکن هر تکه: کمینه {min(_tok_counts)} | میانگین {np.mean(_tok_counts):.1f} | '
      f'بیشینه {max(_tok_counts)}')
print(f'   هدف نرم {_soft} توکن | سقف سخت کتابخانه {HARD_TOKEN_LIMIT} | '
      f'تکه‌های زیر {MIN_PHONEME_TOKENS} توکن با همسایه ادغام شده‌اند')
if max(_tok_counts) > HARD_TOKEN_LIMIT:
    print(f'   ⚠️ {sum(1 for t in _tok_counts if t > HARD_TOKEN_LIMIT)} تکه از سقف سخت رد شد — '
          f'کتابخانه ناچار می‌شود خودش بشکند.')
elif max(_tok_counts) > _soft:
    print(f'   ℹ️ بیشینه از هدف نرم کمی بالاتر است چون تکه‌های خیلی کوتاه ادغام شده‌اند '
          f'(هنوز زیر سقف سخت).')

if _sv.get('ref_emb') is not None and Best_of_N > 1:
    print(f'🎯 برای هر تکه {Best_of_N} نمونه تولید و شبیه‌ترین انتخاب می‌شود '
          f'(~{Best_of_N}× زمان بیشتر).')
else:
    print('ℹ️ انتخاب بهترین نمونه خاموش است (Best_of_N=1 یا سنجش شباهت در دسترس نیست).')

print()
print('🔍 پیش‌نمایش سه تکه‌ی اول:')
for j in JOB_QUEUE[:3]:
    print(f'   «{j["text"][:50]}»')
    print(f'   → {j["ph"][:70]}  [{count_tokens(j["ph"])} توکن]')

os.makedirs(SEG_DIR, exist_ok=True)
for f in os.listdir(SEG_DIR):
    if f.startswith('seg_'):
        os.remove(os.path.join(SEG_DIR, f))

seg_paths, failures, report = [], [], []

print()
for i, job in enumerate(JOB_QUEUE):
    seg = os.path.join(SEG_DIR, f'seg_{i+1:05d}.wav')
    preview = job['text'][:40]

    for attempt in range(1, 5):          # runaway تصادفی است؛ ۴ تلاش
        try:
            a, sim, got = synth(job['ph'], seg)
            if Trim_Segment_Silence:
                a = trim_tail_only(a)
            a = _fade(a, Edge_Fade_ms)
            write_wav(seg, a)
            seg_paths.append(seg)
            report.append((i + 1, job['para'] + 1, len(a) / SAMPLE_RATE, sim, got, preview))
            tag = f' | شباهت {sim:.3f} (از {got})' if sim is not None and got > 1 else (
                  f' | شباهت {sim:.3f}' if sim is not None else '')
            print(f'   ✅ [{i+1}/{len(JOB_QUEUE)}] {len(a)/SAMPLE_RATE:.1f}s{tag} — {preview}...')
            if Similarity_Gate > 0 and sim is not None and sim < Similarity_Gate:
                print(f'      ⚠️ زیر آستانه‌ی {Similarity_Gate:.2f}')
            break
        except Exception as e:
            if attempt == 4:
                failures.append((job['para'] + 1, job['text'], repr(e)))
                print(f'   ❌ [{i+1}/{len(JOB_QUEUE)}] پس از ۴ تلاش رد شد: {e}')
                traceback.print_exc(limit=1)
            else:
                print(f'   ↻ [{i+1}] تلاش {attempt} ناموفق ({type(e).__name__}) — تلاش مجدد...')

    if GPU_OK and i % 20 == 19:
        torch.cuda.empty_cache()

print()
print(f'🎬 ساخت تراک نهایی از {len(seg_paths)} تکه ...')
output_file = concat_segments(seg_paths, OUTPUT_FILE)

if report:
    sims = [r[3] for r in report if r[3] is not None]
    if sims:
        with open(REPORT_FILE, 'w', encoding='utf-8') as f:
            f.write('segment,paragraph,duration_s,similarity,candidates,text\n')
            for r in report:
                s = '' if r[3] is None else f'{r[3]:.4f}'
                f.write(f'{r[0]},{r[1]},{r[2]:.2f},{s},{r[4]},"{r[5].replace(chr(34), chr(39))}"\n')
        print()
        print('📊 شباهت گوینده‌ی تکه‌ها:')
        print(f'   میانگین {np.mean(sims):.3f} | میانه {np.median(sims):.3f} | '
              f'کمینه {np.min(sims):.3f} | بیشینه {np.max(sims):.3f}')
        print(f'   گزارش کامل: {REPORT_FILE}')

if failures:
    with open(FAIL_FILE, 'w', encoding='utf-8') as f:
        for idx, txt, err in failures:
            f.write(f'[پاراگراف {idx}] {err}\n{txt}\n\n')
    print(f'⚠️ {len(failures)} تکه خوانده نشد — جزئیات در {FAIL_FILE}')
    for idx, txt, err in failures[:5]:
        print(f'   • [پاراگراف {idx}] {txt[:60]}... → {err}')

size_mb = os.path.getsize(output_file) / 1e6
total_dur = probe_duration(output_file)
print(f'🎉 تراک نهایی: {output_file} ({size_mb:.1f} MB، {total_dur:.1f} ثانیه)')

# شباهت کل تراک با مرجع — معیار نهایی، نه میانگین تکه‌ها
final_sim = similarity_to_reference(output_file)
if final_sim is not None:
    print(f'🎯 شباهت گوینده‌ی کل تراک با مرجع: {final_sim:.3f}')

# ───────────────────────── پخش و دانلود
display(Audio(output_file, autoplay=True))

btn_download = widgets.Button(description='⬇️ دانلود تراک نهایی', button_style='success')
btn_download.on_click(lambda b: files.download(output_file))
display(btn_download)

if failures:
    btn_fail = widgets.Button(description='⬇️ دانلود لیست تکه‌های ناموفق', button_style='warning')
    btn_fail.on_click(lambda b: files.download(FAIL_FILE))
    display(btn_fail)

if report and os.path.exists(REPORT_FILE):
    btn_rep = widgets.Button(description='⬇️ دانلود گزارش شباهت', button_style='info')
    btn_rep.on_click(lambda b: files.download(REPORT_FILE))
    display(btn_rep)

display(HTML(
    f'<a href="{YOUTUBE_CHANNEL_URL}" target="_blank" '
    f'style="display:inline-block;margin-top:10px;padding:10px 22px;background:#FF0000;'
    f'color:#fff;border-radius:6px;text-decoration:none;font-family:sans-serif;font-weight:bold;">'
    f'🔔 ما را در یوتیوب دنبال کنید</a>'
))


In [ ]:
#@title ✏️ اصلاح دستی تلفظ (جایگزینی در فایل و در حافظه)

import os
import ipywidgets as widgets
from IPython.display import display

if 'SELECTED_DOC_PATH' not in globals() or not SELECTED_DOC_PATH or not os.path.exists(SELECTED_DOC_PATH):
    raise RuntimeError('❌ ابتدا سلول «انتخاب فایل متنی» را اجرا کنید!')

CURRENT_FILE = SELECTED_DOC_PATH
FILE_EXT = CURRENT_FILE.rsplit('.', 1)[-1].lower()


def _replace_in_paragraph(p, w, c):
    full = p.text
    n = full.count(w)
    if not n:
        return 0

    hit = False
    for run in p.runs:
        if w in run.text:
            run.text = run.text.replace(w, c)
            hit = True

    if not hit:
        for run in list(p.runs)[1:]:
            run.text = ''
        if p.runs:
            p.runs[0].text = full.replace(w, c)
    return n


def replace_in_file(w, c):
    total = 0

    if FILE_EXT == 'docx':
        import docx
        doc = docx.Document(CURRENT_FILE)
        for p in doc.paragraphs:
            total += _replace_in_paragraph(p, w, c)
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    for p in cell.paragraphs:
                        total += _replace_in_paragraph(p, w, c)
        if total:
            doc.save(CURRENT_FILE)
        return total

    with open(CURRENT_FILE, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.read().splitlines()

    new_lines = []
    for line in lines:
        total += line.count(w)
        new_lines.append(line.replace(w, c))

    if total:
        with open(CURRENT_FILE, 'w', encoding='utf-8') as f:
            f.write('\n'.join(new_lines))
    return total


wrong_word = widgets.Text(description='کلمه اشتباه:', placeholder='مثلاً: نکته',
                          layout=widgets.Layout(width='460px'))
correct_word = widgets.Text(description='عبارت درست:', placeholder='مثلاً: نُکته',
                            layout=widgets.Layout(width='460px'))
btn_replace = widgets.Button(description='🔁 جایگزینی', button_style='warning')
btn_preview = widgets.Button(description='🔍 پیش‌نمایش فونم', button_style='info')
log_output = widgets.Output()

display(widgets.HTML(f'<b>📄 فایل فعلی:</b> {CURRENT_FILE}'),
        wrong_word, correct_word, widgets.HBox([btn_replace, btn_preview]), log_output)


def on_replace_clicked(b):
    with log_output:
        log_output.clear_output()

        w = wrong_word.value.strip()
        c = correct_word.value.strip()

        if not w or not c:
            print('⚠️ هر دو فیلد را پر کنید.')
            return
        if w == c:
            print('⚠️ عبارت اشتباه و درست یکسان‌اند.')
            return

        n_file = replace_in_file(w, c)

        n_mem = 0
        if 'PARAGRAPHS' in globals() and PARAGRAPHS:
            for i, p in enumerate(PARAGRAPHS):
                if w in p:
                    n_mem += p.count(w)
                    PARAGRAPHS[i] = p.replace(w, c)

        if n_file == 0 and n_mem == 0:
            print(f'ℹ️ عبارت «{w}» پیدا نشد.')
            return

        print(f'✅ {n_file} مورد در فایل و {n_mem} مورد در حافظه جایگزین شد.')
        print(f'🔍 فونم جدید: {phonemise(c)}')
        wrong_word.value = ''
        correct_word.value = ''


def on_preview_clicked(b):
    with log_output:
        log_output.clear_output()
        t = wrong_word.value.strip()
        if not t:
            print('⚠️ متنی برای پیش‌نمایش بنویسید.')
            return
        print(f'«{t}»')
        print(f'→ {phonemise(t)}')
        print()
        print('اگر فونم خالی است، G2P نتوانسته بخواند — شکل دیگری بنویسید.')


btn_replace.on_click(on_replace_clicked)
btn_preview.on_click(on_preview_clicked)

print('راهنما: کلمه‌ی اشتباه‌خوانده‌شده را در فیلد اول و شکل درست (با اعراب/نیم‌فاصله) را در فیلد دوم بنویس. '
      'با «🔍 پیش‌نمایش فونم» ببین G2P چه خوانشی می‌دهد، بعد «🔁 جایگزینی» را بزن. '
      'در پایان سلول «▶️ شروع فرآیند» را دوباره اجرا کن.')


In [ ]:
#@title 🧪 (اختیاری) تست دستی بدون فایل — وارد کردن متن مستقیم

# این سلول از توابع سلول «▶️ شروع فرآیند» استفاده می‌کند،
# پس یک بار آن سلول را اجرا کنید (حتی اگر فایل ندارید).

import os
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, Audio

Manual_Text = "" #@param {type:"string"}
YOUTUBE_CHANNEL_URL = "https://www.youtube.com/@aigolden?sub_confirmation=1"

if 'synth' not in globals() or 'build_chunks' not in globals():
    raise RuntimeError('❌ ابتدا یک بار سلول «▶️ شروع فرآیند» را اجرا کنید تا توابع آماده شوند.')
if 'voice_state' not in globals():
    raise RuntimeError('❌ ابتدا سلول «آماده‌سازی صدای مرجع» را اجرا کنید!')

paragraphs = [l.strip() for l in Manual_Text.splitlines() if l.strip()]
if not paragraphs:
    raise RuntimeError('❌ متنی وارد نشده است!')

MANUAL_SEG_DIR = 'voice_clone_segments_manual'
os.makedirs(MANUAL_SEG_DIR, exist_ok=True)

jobs = []
for para in paragraphs:
    jobs.extend(build_chunks(para))

if not jobs:
    raise RuntimeError('❌ فونمی تولید نشد — متن فارسی است؟')

print(f'🔍 {len(jobs)} تکه:')
for txt, ph in jobs:
    print(f'   «{txt[:45]}»  →  {ph[:60]}  [{count_tokens(ph)} توکن]')

seg_paths, failures, sims = [], [], []
for i, (txt, ph) in enumerate(jobs):
    seg = os.path.join(MANUAL_SEG_DIR, f'seg_{i+1:05d}.wav')
    for attempt in range(1, 5):
        try:
            a, sim, got = synth(ph, seg)
            if Trim_Segment_Silence:
                a = trim_tail_only(a)
            a = _fade(a, Edge_Fade_ms)
            write_wav(seg, a)
            seg_paths.append(seg)
            if sim is not None:
                sims.append(sim)
            tag = f' | شباهت {sim:.3f}' if sim is not None else ''
            print(f'   ✅ [{i+1}/{len(jobs)}] {len(a)/SAMPLE_RATE:.1f}s{tag} — {txt[:40]}...')
            break
        except Exception as e:
            if attempt == 4:
                failures.append((i + 1, txt, repr(e)))
                print(f'   ❌ [{i+1}] رد شد: {e}')
            else:
                print(f'   ↻ [{i+1}] تلاش {attempt} ناموفق — تلاش مجدد...')

output_file = concat_segments(seg_paths, 'cloned_voice_manual_test.mp3', work_dir=MANUAL_SEG_DIR)

print(f'🎉 تراک آزمایشی آماده شد: {output_file}')
if sims:
    import numpy as np
    print(f'📊 شباهت گوینده: میانگین {np.mean(sims):.3f} | کمینه {np.min(sims):.3f} | بیشینه {np.max(sims):.3f}')
final_sim = similarity_to_reference(output_file)
if final_sim is not None:
    print(f'🎯 شباهت کل تراک با مرجع: {final_sim:.3f}')
if failures:
    print(f'⚠️ {len(failures)} تکه رد شد.')

display(Audio(output_file, autoplay=True))

btn_download = widgets.Button(description='⬇️ دانلود تراک آزمایشی', button_style='success')
btn_download.on_click(lambda b: files.download(output_file))
display(btn_download)

display(HTML(
    f'<a href="{YOUTUBE_CHANNEL_URL}" target="_blank" '
    f'style="display:inline-block;margin-top:10px;padding:10px 22px;'
    f'background:#FF0000;color:#fff;border-radius:6px;'
    f'text-decoration:none;font-family:sans-serif;font-weight:bold;">'
    f'🔔 ما را در یوتیوب دنبال کنید</a>'
))


In [ ]:
#@title 🧹 پاکسازی فایل‌های موقت

import os, glob, shutil

#@markdown فایل‌های mp3 و پوشه‌ی Drive دست‌نخورده می‌مانند.
Wipe_Everything = False #@param {type:"boolean"}

KEEP_SUFFIXES = ('.mp3',)

removed = []

for pattern in ('/content/voice_clone_segments*', '/content/ref_voice.*',
                '/content/_silence.wav', '/content/_list.txt',
                '/content/failed_paragraphs.txt'):
    for p in glob.glob(pattern):
        if os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)
        else:
            try:
                os.remove(p)
            except OSError:
                continue
        removed.append(p)

if Wipe_Everything:
    for entry in os.listdir('/content'):
        if entry == 'drive':
            continue
        if entry.lower().endswith(KEEP_SUFFIXES):
            continue
        full = os.path.join('/content', entry)
        try:
            shutil.rmtree(full) if os.path.isdir(full) else os.remove(full)
            removed.append(full)
        except OSError:
            pass

print('🧹 حذف شد:' if removed else 'ℹ️ چیزی برای حذف نبود.')
for p in removed:
    print('  •', p)


## ⚠️ راهنمای این نسخه (pocket-tts-farsi-v2) — v3

### مدل و داده تغییر نکرده

همان `mehdi-hf/pocket-tts-farsi-v2`، همان وزن‌ها، همان G2P، همان فایل‌های ورودی شما.
فقط مسیر کد عوض شده. ۱۰۹.۵ میلیون پارامتر، آموزش روی **۹۷۳ ساعت** گفتار فارسی از **۲۹۷۸ گوینده**.
اعداد روی ۳۰۰ نمونه‌ی Common Voice که مدل ندیده:

| معیار | v1 | **v2** |
|---|---|---|
| WER میانگین | 2.048 | **0.582** |
| شباهت گوینده | 0.764 | **0.859** |
| UTMOS (طبیعی‌بودن) | 2.826 | **3.11** |
| runaway | ۲۵ از ۳۰۰ | **۰–۱ از ۳۰۰** |

---

## چهار اشکال واقعی که پیدا و رفع شد

### ۱. تقسیم‌کننده‌ی جمله‌ی کتابخانه، فونم را خراب می‌کرد — بحرانی‌ترین مورد

`generate_audio` رشته را از `split_into_best_sentences` عبور می‌دهد و آن تابع روی نویسه‌های `.!...?` می‌شکند. در این مدل `?` **انسدادی چاکنایی (همزه)** است و وسط کلمه هم می‌آید.

اندازه‌گیری روی تابع واقعی کتابخانه (`pocket_tts/models/text_chunking.py`) با توکنایزر واقعی مدل:

| فونم ورودی | خروجی مسیر قدیم | وضعیت |
|---|---|---|
| `so?Ale ?asli ?in ?ast` | `so? Ale ?asli ?in ?ast` | خراب |
| `ta?sire ?in tasmim ziyAd ?ast` | `ta? sire ?in tasmim ziyAd ?ast` | خراب |
| `mo?assese ?eqtesAdi goft` | `mo? assese ?eqtesAdi goft` | خراب |
| `pa;uheSgarAn mo?taqedand` | `pa;uheSgarAn mo? taqedand` | خراب |
| `mardom ?entezAr dArand` | `mardom ?entezAr dArand` | سالم |

روی ۶ جمله‌ی فارسی واقعی (خروجی G2P واقعی، نه ساختگی): **۶ از ۶ جمله خراب می‌شد** و **۱۳ از ۵۷ کلمه (۲۲.۸٪)** همزه‌ی میانی داشتند.

این نسخه رشته‌ی فونم را دست‌نخورده به مسیر تولید می‌دهد (`generate_exact` → `_generate_audio_stream_short_text` که فقط `conditioner.prepare` را صدا می‌زند).

**توجه:** این باگ در نسخه‌ی قبلی که تحویل داده بودم هم بود، چون از `generate_audio` استاندارد استفاده می‌کرد.

### ۲. سقف طول با واحد اشتباه شمرده می‌شد

`Max_Phoneme_Tokens` روی **تعداد کلمه** حساب می‌شد، ولی کتابخانه توکن sentencepiece می‌شمارد. اندازه‌گیری روی ۶ جمله‌ی واقعی: **میانگین ۲.۰۱۳ توکن به ازای هر کلمه** (میانه ۲.۱۱، دامنه ۱.۴۰ تا ۲.۳۳).

پس «۱۸ کلمه» در واقع تا **~۳۶ توکن** بود، نه ۱۸. بدتر: اگر یک واحد از سقف بلندتر بود، با `cur_n == 0` به‌هرحال یک‌جا اضافه می‌شد و کتابخانه ناچار بود خودش بشکند — که مستقیم به باگ ۱ می‌رسید.

حالا شمارش با توکن واقعی مدل است و واحدهای بلند با `_hard_split` روی مرز کلمه به تکه‌های **هم‌اندازه** بریده می‌شوند (برش حریصانه یک تکه‌ی ۱ توکنی باقی می‌گذاشت و مدل روی تکه‌ی خیلی کوتاه، پرامپت را ادامه می‌دهد).

آزمون روی یک پاراگراف ۸۴ کلمه‌ای واقعی:

| بودجه | تکه‌ها | توکن‌ها | زیر ۶ توکن | توالی فونم حفظ شد؟ |
|---|---|---|---|---|
| ۲۴ | ۱۰ | ۱۲..۲۲ | ۰ | ✅ |
| ۱۸ | ۱۲ | ۸..۲۱ | ۰ | ✅ |
| ۱۲ | ۱۷ | ۷..۱۶ | ۰ | ✅ |

و یک جمله‌ی ۴۱ کلمه‌ای بدون هیچ نشانه‌گذاری: → ۴ تکه‌ی ۲۴/۲۴/۲۴/۲۳ توکن، توالی دست‌نخورده.

### ۳. آستانه‌ی برش سکوت، گفتار واقعی را می‌بُرید

آستانه‌ی ثابت −۳۵ dB روی فایل‌های رسمی خود مدل، **۱.۳۲ ثانیه از انتهای `prompt_hello`** را می‌برد که گفتار واقعی است. حالا آستانه **نسبی** است (صدک ۹۵ انرژی فریم‌های همان کلیپ منهای `Silence_Rel_dB`):

| فایل | طول | ثابت −۳۵ dB | نسبی ۳۵ dB |
|---|---|---|---|
| `prompt_hello` | 4.30s | 2.98s ⚠️ | **4.30s** ✅ |
| `prompt_news_paragraph` | 5.14s | 3.27s | **3.87s** |
| `prompt_short_sentence` | 4.94s | 3.23s | **3.83s** |

### ۴. `Frames_After_Eos = 0` دنباله‌ی واکه را می‌بُرید

حلقه‌ی `_autoregressive_generation` شرط `generation_step >= eos_step + frames_after_eos` دارد. با صفر، توقف **در همان فریم EOS** رخ می‌دهد. کتابخانه در حالت خودکار `3 if words <= 4 else 1` را حساب می‌کند و **۲** هم اضافه می‌کند؛ مقدار صریح، این حدس را دور می‌زند.

این نسخه `-1` را به‌عنوان «خودکار» می‌پذیرد و همان محاسبه‌ی کتابخانه را انجام می‌دهد.

---

## سه چیزی که باید بدانید

**۱. صدای مرجع حداکثر ۵ ثانیه.** مهم‌ترین قاعده. مدل روی پرامپت ≤۵ ثانیه آموزش دیده؛ بلندتر باعث می‌شود مدل جمله‌ی خودِ پرامپت را ادامه دهد نه متن شما. اندازه‌گیری رسمی: ۶.۱ ثانیه → ۰ از ۴ بار؛ همان کلیپ در ۵ ثانیه → ۴ از ۴ بار.

**۲. مدل خط فارسی نمی‌خواند.** ورودی‌اش فونم رومانیزه است. متن فارسی مستقیم = **سکوت مطلق، بدون هیچ خطا**. برای همین سلول «🧠 لود مدل» یک موتور G2P (`Homo-GE2PE-Persian-HF`) هم لود می‌کند.

**۳. سقف ۵۰ توکن در خود کتابخانه.** `MAX_TOKEN_PER_CHUNK = 50`. این نسخه پیش‌فرض را روی ۲۴ می‌گذارد و هرگز عمداً از ۵۰ رد نمی‌کند.

---

## الفبای فونم مدل

واژگان ۴۰۰۰ تکه‌ای مدل فقط این ۳۳ نویسه را می‌شناسد (استخراج‌شده از `tokenizer_ph.model`):

```
- ; ? A C S a b d e f g h i j k l m n o p q r s t u v x y z
```

`?` همزه، `;` ژ، `S` ش، `C` چ، `A` آ. هر نویسه‌ی دیگری (`,`، `.`، `!`، `:`، حروف بزرگ دیگر، `_`، `c`، `w`) به `<unk>` تبدیل می‌شود — `sanitise_phonemes` آن‌ها را حذف می‌کند. خروجی G2P دقیقاً همین مجموعه است، پس در عمل هیچ `<unk>` ساخته نمی‌شود.

---

## سنجش شباهت: چه چیزی کار می‌کند و چه چیزی نه

`Best_of_N` برای هر تکه چند نمونه تولید می‌کند و شبیه‌ترین به مرجع را نگه می‌دارد. معیار، بردار WavLM-SV است (۵۱۲ بعد، نرمال‌شده).

**آنچه اندازه‌گیری شد:** روی فایل‌های رسمی، پراکندگی درون‌گوینده‌ی همین معیار بزرگ است — روی پنجره‌های مختلف `hello.wav`، شباهت جفتی از ۰.۶۸۷ تا ۰.۹۸۹ (دامنه ۰.۳۰) تغییر می‌کند. پس انتخاب بر اساس این معیار **بی‌اثر نیست**.

**آنچه اندازه‌گیری شد و رد شد:** انتخاب «بهترین پنجره» از یک فایل بلند بر اساس بردار گوینده (روش مدوید). روی یک فایل ۱۴ ثانیه‌ای تک‌گوینده، رتبه‌ی شباهت‌به‌هویت‌کل‌فایل از ۱۹ پنجره:

- پنجره‌ی اول: **رتبه ۲** (cos 0.9969)
- بیشترین نسبت گفتار: رتبه ۷ (cos 0.9602)
- مدوید بردار گوینده: رتبه ۷ (cos 0.9602)
- همبستگی نسبت‌گفتار ↔ شباهت: **+۰.۰۸۰** (یعنی تقریباً هیچ)

⇒ این نسخه فقط **نسبت گفتار** را ملاک می‌گذارد (تا پنجره‌ی سکوت‌محور انتخاب نشود) و ادعای «نماینده‌ترین پنجره» نمی‌کند.

---

## runaway چیست؟

تولیدی که هرگز `end-of-speech` نمی‌دهد و تا سقف طول تکرار می‌کند — `fanAvari` می‌شود `fanAvariiiiii…`. تصادفی است و تلاش دوم معمولاً تمام می‌شود. کد مدت خروجی را با `توکن ÷ ۳ + ۲` ثانیه مقایسه می‌کند و اگر بیش از ۲.۲ برابر بود، دوباره تولید می‌کند (تا ۴ تلاش).

---

## محدودیت‌های شناخته‌شده

- **کلمه‌ی اول هر تکه** ضعیف‌ترین نقطه است: حدود ۴۰ بار از ۵۰ درست. حرف انسدادی آغازین می‌لرزد (`کریس` → گاهی `پریس`). به همین دلیل برش سکوت فقط از **انتها** انجام می‌شود، نه از ابتدا.
- **هیچ نشانه‌گذاری‌ای در واژگان مدل نیست.** نمی‌توانید با کاما مکث بخواهید؛ تنها اهرم شما مکث بین تکه‌هاست.
- **WER مطلق بالا است** چون Whisper خودش روی Common Voice خطای ۰.۲۷۳ دارد. این اعداد برای مقایسه‌ی مدل‌ها هستند، نه نمره‌ی کیفیت مطلق.
- **مجوز CC-BY-NC-4.0** — غیرتجاری.

### آنچه اندازه‌گیری نشده

این‌ها در محیط توسعه‌ی این نسخه قابل اندازه‌گیری نبودند (سقف حافظه‌ی سندباکس اجازه‌ی لود مدل ۱۰۹.۵M را نداد). صریحاً «ادعا» هستند، نه «نتیجه»:

- **مقدار بهبود شباهت از `Best_of_N`** روی گفتار واقعی. منطق درست است (بیشینه‌ی N نمونه هرگز بدتر از نمونه‌ی اول نیست) ولی عدد بهبود سنجیده نشد.
- **مقدار بهبود شباهت از `Temperature` پایین‌تر** (۰.۲–۰.۲۵). منطق نمونه‌برداری است، بدون اندازه‌گیری.
- **اثر شنیداری رفع باگ همزه.** تغییر رشته اثبات‌شده است (۶/۶ جمله)، اثر ادراکی شنیده نشد.
- **اثر شنیداری رفع `frames_after_eos`.** استنتاج از کد `break` است، بدون شنیدن خروجی.
- **رندر شدن نوت‌بوک در Colab واقعی.** همه‌ی سلول‌ها با stub و mock اجرا شدند، نه روی GPU واقعی Colab.

---

## اگر مجوز تجاری می‌خواهید

`mehdi-hf/chatterbox-farsi` (مجوز MIT، خط فارسی را مستقیم می‌خواند، WER ۱۲.۵٪ زن / ۱۶.۹٪ مرد). نیازی به G2P ندارد ولی ۵ برابر بزرگ‌تر است و روی CPU کند.

## تفاوت با نسخه‌ی OmniVoice

| | OmniVoice | pocket-tts-farsi-v2 |
|---|---|---|
| اندازه | ۳.۲۷ GB | ~۴۴۰ MB |
| ورودی | خط فارسی مستقیم | فونم (نیاز به G2P) |
| WER فارسی | منتشر نشده | **0.582** |
| سرعت CPU | کند | **~۶× سریع‌تر از زمان واقعی** |
| مجوز | — | CC-BY-NC-4.0 |
